# Load the Olist marketplace data into PostgreSQL

About 100k orders from Olist, a Brazilian marketplace, from 2016 to 2018. The data comes as 9 linked CSV
files. This notebook creates a relational schema with primary and foreign keys (`sql/schema.sql`), bulk-
loads 8 of the tables with `COPY`, and checks that everything arrived intact. The 1M-row geolocation
table is not needed for these questions, so it is skipped.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
SQL = ROOT / "sql"

load_dotenv(ROOT / ".env")
engine = create_engine(os.environ["DATABASE_URL"])

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database()")).scalar())

olist_db


In [2]:
with engine.begin() as conn:
    conn.execute(text((SQL / "schema.sql").read_text()))

pd.read_sql("""
SELECT table_name, COUNT(*) AS columns
FROM information_schema.columns
WHERE table_schema = 'public'
GROUP BY table_name
ORDER BY table_name
""", engine)

,table_name,columns
0,category_translation,2
1,customer_rfm,11
2,customers,5
3,first_orders,20
4,order_facts,23
5,order_items,7
6,order_payments,5
7,order_reviews,7
8,orders,8
9,products,9


Load parent tables before child tables so every foreign key has something to point at.

In [3]:
import time

LOAD_ORDER = {
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "products": "olist_products_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
}

start = time.time()
raw = engine.raw_connection()
try:
    with raw.cursor() as cur:
        for table, file in LOAD_ORDER.items():
            with open(RAW / file, encoding="utf-8-sig") as f:
                cur.copy_expert(f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER true)", f)
    raw.commit()
finally:
    raw.close()
print(f"Loaded in {time.time() - start:.1f} seconds")

Loaded in 21.4 seconds


In [4]:
counts = []
for table, file in LOAD_ORDER.items():
    in_csv = len(pd.read_csv(RAW / file))
    in_db = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", engine)["n"].iloc[0]
    counts.append({"table": table, "csv_rows": in_csv, "db_rows": in_db, "match": in_csv == in_db})

counts = pd.DataFrame(counts)
assert counts["match"].all()
counts

,table,csv_rows,db_rows,match
0,customers,99441,99441,True
1,sellers,3095,3095,True
2,category_translation,71,71,True
3,products,32951,32951,True
4,orders,99441,99441,True
5,order_items,112650,112650,True
6,order_payments,103886,103886,True
7,order_reviews,99224,99224,True


### Data quality checks
The foreign keys already guarantee there are no orphan rows. These queries look for the quirks that
matter for the analysis.

In [5]:
pd.read_sql("""
SELECT
    (SELECT COUNT(*) FROM customers) AS customer_ids,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM customers) AS unique_customers,
    (SELECT COUNT(*) FROM orders o WHERE NOT EXISTS (SELECT 1 FROM order_items i WHERE i.order_id = o.order_id))
        AS orders_without_items,
    (SELECT COUNT(*) FROM orders o WHERE NOT EXISTS (SELECT 1 FROM order_reviews r WHERE r.order_id = o.order_id))
        AS orders_without_review,
    (SELECT COUNT(*) FROM (SELECT order_id FROM order_reviews GROUP BY order_id HAVING COUNT(*) > 1) t)
        AS orders_with_several_reviews,
    (SELECT COUNT(*) FROM orders WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL)
        AS delivered_without_date
""", engine).T.rename(columns={0: "count"})

,count
customer_ids,99441
unique_customers,96096
orders_without_items,775
orders_without_review,768
orders_with_several_reviews,547
delivered_without_date,8


In [6]:
pd.read_sql("""
SELECT
    order_status,
    COUNT(*) AS orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM orders
GROUP BY order_status
ORDER BY orders DESC
""", engine)

,order_status,orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [7]:
pd.read_sql("""
SELECT
    DATE_TRUNC('month', order_purchase_timestamp)::date AS month,
    COUNT(*) AS orders
FROM orders
GROUP BY 1
ORDER BY 1
""", engine)

,month,orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


### Findings
- All 8 tables loaded with row counts matching the CSVs exactly, and every foreign key holds.
- **customer_id is not a customer.** Olist issues a new `customer_id` for every order. The 99,441
  customer IDs belong to **96,096 real people** (`customer_unique_id`). Any retention analysis must use
  `customer_unique_id`, or every customer looks brand new on every order.
- 97% of orders were delivered. 775 orders (mostly canceled or unavailable) have no items, 768 have no
  review, and 547 have more than one review, so the analysis keeps the latest.
- Orders run from September 2016 to October 2018, but 2016 and September-October 2018 have only a
  handful of orders each. Time-based analysis uses **January 2017 to August 2018**.